<a href="https://colab.research.google.com/github/Shirish-12105/AI-HIRING-PREDICTION-SYSTEM/blob/main/ai_hirinig_prediction_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ================= IMPORTS =================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ================= LOAD DATA =================
df = pd.read_csv('/content/AI-Based Hiring Prediction System.csv')

# ================= CLEANING =================
df = df.drop(columns=['Resume_ID', 'Name', 'AI Score (0-100)'])

df['Recruiter Decision'] = df['Recruiter Decision'].map({'Hire': 1, 'Reject': 0})

df['Skills'] = df['Skills'].fillna('')
df['Certifications'] = df['Certifications'].fillna('')
df['Job Role'] = df['Job Role'].fillna('')

# ================= TEXT PROCESSING =================
df['combined_text'] = (
    df['Skills'] + " " +
    df['Certifications'] + " " +
    df['Job Role']
).str.lower()

df['combined_text'] = df['combined_text'].str.replace(r'[^a-z\s]', '', regex=True)

tfidf = TfidfVectorizer(max_features=500)
X_text = tfidf.fit_transform(df['combined_text']).toarray()

# ================= ENCODING =================
le = LabelEncoder()
df['Education_Encoded'] = le.fit_transform(df['Education'])

# ================= FEATURES =================
X_num = df[['Experience (Years)', 'Salary Expectation ($)', 'Projects Count', 'Education_Encoded']].values
y = df['Recruiter Decision'].values

X = np.hstack((X_num, X_text))

# ================= SPLIT =================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ================= SCALING =================
scaler = StandardScaler()

# scale only numeric (first 3 columns: exp, salary, projects)
X_train[:, :3] = scaler.fit_transform(X_train[:, :3])
X_test[:, :3] = scaler.transform(X_test[:, :3])

# ================= MODEL =================
model = RandomForestClassifier()
model.fit(X_train, y_train)

# ================= EVALUATION =================
preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

# ================= FIXED PREDICTION FUNCTION =================
def predict_hiring(skills, exp, edu, cert, role, projects, salary):

    # text
    combined = (str(skills) + " " + str(cert) + " " + str(role)).lower()
    text_vec = tfidf.transform([combined])

Accuracy: 0.95
              precision    recall  f1-score   support

           0       1.00      0.78      0.88        46
           1       0.94      1.00      0.97       154

    accuracy                           0.95       200
   macro avg       0.97      0.89      0.92       200
weighted avg       0.95      0.95      0.95       200

